# Stage 0 — pdfplumber Text Extraction

**Why this approach:**  
The original pypdf approach (see `stage0_exploration.ipynb`) had three problems:
1. Header/footer text (company name, date, page number) injected mid-sentence across page boundaries
2. No paragraph spacing preserved within a speaker's comment
3. Analyst names missed by regex because pypdf loses column layout

**This approach:**
1. `pdfplumber` with `layout=True` — preserves paragraph spacing, better text ordering
2. Strip company name / date / page number per page before concatenating (they appear as standalone lines)
3. Detect all speaker names from `Name:` patterns in the clean text — no coordinate tracking
4. Parse management names from page 2 participant list for role classification
5. Split full text by speaker names → ordered `(speaker, role, comment)` turns

## Setup

In [ ]:
import re
import pdfplumber
from collections import Counter

PDF_PATH = "transcripts/fineotex_chemical_Q4_FY26.pdf"
# PDF_PATH = "transcripts/asian_paints_Q4_FY26.pdf"
# PDF_PATH = "transcripts/sandhar_technologies_Q4_FY26.pdf"
# PDF_PATH = "transcripts/mold-tek_packaging_Q4_FY26.pdf"

pdf = pdfplumber.open(PDF_PATH)
print(f"PDF   : {PDF_PATH}")
print(f"Pages : {len(pdf.pages)}")

---
## Step 1 — Extract management names from page 2

Page 2 has the participant list: `MANAGEMENT: MS. NAME – TITLE`.  
Parse the ALL-CAPS names and convert to Title Case to match how they appear in the transcript body.

In [ ]:
# Inspect raw page 2 text to understand the participant list format
page2_text = pdf.pages[1].extract_text(layout=True) or ""
print(page2_text)

In [ ]:
# Parse: MR./MS./MRS./DR. followed by ALL-CAPS name, then dash and title
_MGMT_NAME_RE = re.compile(
    r"(?:MR|MS|MRS|DR)\.\s+([A-Z][A-Z.\s]+?)\s+[–\-]",
    re.IGNORECASE
)

def extract_management_names(page2_text):
    m = re.search(r"MANAGEMENT\s*:(.*)", page2_text, re.S | re.I)
    if not m:
        print("WARNING: No MANAGEMENT: block found on page 2")
        return set()
    mgmt_block = m.group(1)
    names = set()
    for raw in _MGMT_NAME_RE.findall(mgmt_block):
        name = " ".join(raw.split()).title()   # collapse whitespace + Title Case
        names.add(name)
    return names


mgmt_names = extract_management_names(page2_text)

print(f"Management names ({len(mgmt_names)}):")
for name in sorted(mgmt_names):
    print(f"  {name}")

---
## Step 2 — Extract raw text from content pages

`extract_text(layout=True)` preserves paragraph gaps as blank lines and keeps indentation.  
Content starts at page 3 (pages 1–2 are cover letter and participant list).  

Inspect raw output before any cleaning.

In [ ]:
CONTENT_START_PAGE = 3   # 1-indexed — first page with speaker turns

raw_pages = []
for page in pdf.pages[CONTENT_START_PAGE - 1:]:
    text = page.extract_text(layout=True) or ""
    raw_pages.append(text)

print(f"Content pages: {len(raw_pages)}")
print(f"Total chars  : {sum(len(p) for p in raw_pages):,}")

In [ ]:
# Inspect one raw page — look at header at the top and footer at the bottom
PAGE_TO_INSPECT = 0   # 0 = first content page

print(f"=== Raw page {CONTENT_START_PAGE + PAGE_TO_INSPECT} (before cleaning) ===")
print(raw_pages[PAGE_TO_INSPECT])

---
## Step 3 — Strip header / footer from each page

Two removal strategies combined:
- **Full-line removal** — company name, date, `Page N of M` when they occupy the entire line (so a company name inside a sentence is never removed)
- **Inline removal** — `Page N of M` also stripped when it appears at the end of a body text line (pdfplumber sometimes places the footer on the same line as the last sentence of a page)

In [ ]:
_MONTHS = (
    "January|February|March|April|May|June|"
    "July|August|September|October|November|December"
)
_DATE_RE            = re.compile(rf"^({_MONTHS})\s+\d{{1,2}},?\s+\d{{4}}$", re.I)
_COMPANY_RE         = re.compile(r"^.+\b(Limited|Ltd\.?|Inc\.?|Corp\.?|Pvt\.?)$", re.I)
_PAGE_NUMBER_RE     = re.compile(r"^Page\s+\d+\s+of\s+\d+$", re.I)
_PAGE_NUMBER_INLINE = re.compile(r"\s*Page\s+\d+\s+of\s+\d+", re.I)  # for inline removal


def strip_header_footer(page_text):
    cleaned = []
    for line in page_text.split("\n"):
        s = line.strip()
        if not s:
            cleaned.append(line)          # keep blank lines — they carry paragraph spacing
        elif _DATE_RE.match(s) or _COMPANY_RE.match(s) or _PAGE_NUMBER_RE.match(s):
            continue                      # full-line header/footer — drop entire line
        else:
            # Inline "Page N of M" at end of a body text line — strip just that fragment
            line = _PAGE_NUMBER_INLINE.sub("", line).rstrip()
            if line.strip():
                cleaned.append(line)
    return "\n".join(cleaned)


# Test on the inspect page
raw   = raw_pages[PAGE_TO_INSPECT]
clean = strip_header_footer(raw)

removed = [l.strip() for l in raw.split("\n") if l.strip() and l not in clean.split("\n")]
print("Lines removed:")
for line in removed:
    print(f"  [{line}]")

In [ ]:
# Confirm the cleaned page looks correct — no header, body text intact
print(f"=== Cleaned page {CONTENT_START_PAGE + PAGE_TO_INSPECT} ===")
print(clean)

In [ ]:
# Apply to all content pages and check another page
clean_pages = [strip_header_footer(p) for p in raw_pages]

CHECK_PAGE = 3   # 0-indexed among content pages — change to inspect any page
print(f"=== Cleaned page {CONTENT_START_PAGE + CHECK_PAGE} ===")
print(clean_pages[CHECK_PAGE])

---
## Step 4 — Concatenate into one clean transcript string

Join all cleaned page texts. A speaker turn that starts at the bottom of one page and continues on the next is naturally joined — no special handling needed.

In [ ]:
full_text = "\n".join(clean_pages)

print(f"Total characters : {len(full_text):,}")
print(f"Total words      : {len(full_text.split()):,}")
print()
print("First 2000 characters:")
print(full_text[:2000])

---
## Step 5 — Detect all speaker names in the text

Speaker headers appear as indented `Name:` at the start of a line — e.g. `          Aarti Jhunjhunwala:`.  
Pattern: 5+ spaces, 1–4 Title Case words (`[A-Z][a-z]+`), colon.  
`[A-Z][a-z]+` rejects ALL-CAPS words like `EBITDA:` or `Q:` which could otherwise be false positives.

**Check:** Do all detected names look like real speakers? Any false positives to exclude?

In [ ]:
_SPEAKER_DETECT_RE = re.compile(
    r"^[ \t]{5,}([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3}):",
    re.MULTILINE
)

detected = Counter(m.group(1).strip() for m in _SPEAKER_DETECT_RE.finditer(full_text))

print(f"Speaker names detected ({len(detected)} unique):")
print()
print(f"  {'turns':>6}  {'in mgmt roster':<16} name")
print("  " + "-" * 50)
for name, count in detected.most_common():
    in_mgmt = "[management]" if name in mgmt_names else ""
    print(f"  {count:>6}  {in_mgmt:<16} {name}")

In [ ]:
# Add any false positives here to exclude them from the roster
EXCLUDE_NAMES = set()   # e.g. {"Note", "Disclaimer"}

speaker_names = sorted(
    name for name in detected if name not in EXCLUDE_NAMES
)

print(f"Final speaker roster ({len(speaker_names)} names): {speaker_names}")

---
## Step 6 — Split text by speaker names → turns

Build one regex from the full roster and split the clean text at every `Name:` occurrence.  
Each segment = one complete turn (the full comment, with paragraph spacing preserved).

In [ ]:
# Longest names first to avoid partial matches (e.g. "Aarti" before "Aarti Jhunjhunwala")
name_alts = "|".join(re.escape(n) for n in sorted(speaker_names, key=len, reverse=True))
_SPLIT_RE = re.compile(rf"^[ \t]*({name_alts}):", re.MULTILINE)

parts = _SPLIT_RE.split(full_text)
# split() with a capture group → [pre_text, name1, block1, name2, block2, ...]

print(f"Parts from split: {len(parts)}")
print(f"Expected turns  : {(len(parts) - 1) // 2}")

if parts[0].strip():
    print(f"\nPre-first-speaker text ({len(parts[0].strip())} chars):")
    print(parts[0].strip()[:300])

In [ ]:
def build_turns(parts, mgmt_names):
    _MOD_RE = re.compile(r"moderator|operator", re.I)

    def role(speaker):
        if _MOD_RE.search(speaker):
            return "moderator"
        if speaker in mgmt_names:
            return "management"
        return "analyst"

    turns = []
    i = 1
    while i + 1 < len(parts):
        speaker = parts[i].strip()
        comment = parts[i + 1].strip()
        if comment:
            turns.append({"speaker": speaker, "role": role(speaker), "comment": comment})
        i += 2
    return turns


turns = build_turns(parts, mgmt_names)

role_counts = Counter(t["role"] for t in turns)
print(f"Total turns : {len(turns)}")
for role, count in role_counts.most_common():
    print(f"  {role:<12} {count}")

---
## Step 7 — Final output

Ordered `(speaker, role, comment)` list. Paragraph spacing within each comment is preserved as blank lines.

In [ ]:
# Overview table — all turns
print(f"  {'#':<5} {'role':<12} {'speaker':<30} {'words'}")
print("  " + "-" * 62)
for i, t in enumerate(turns):
    words = len(t["comment"].split())
    print(f"  {i:<5} {t['role']:<12} {t['speaker']:<30} {words}")

In [ ]:
# Inspect any single turn in full — change TURN_INDEX
TURN_INDEX = 0

t = turns[TURN_INDEX]
print(f"Speaker : {t['speaker']}")
print(f"Role    : {t['role']}")
print(f"Words   : {len(t['comment'].split())}")
print()
print(t["comment"])

In [ ]:
# Print the first 5 turns formatted
for t in turns[:5]:
    print("=" * 70)
    print(f"[{t['role'].upper()}]  {t['speaker']}")
    print("=" * 70)
    print(t["comment"])
    print()

In [ ]:
pdf.close()
print("Done.")